**Last modified on**: 16/04/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

16/04/2026: Initial version

**Credits**:

This Jupyter notebook accompanies the HMMER & Profile Hidden Markov Models lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Eddy SR (2011). Accelerated profile HMM searches. PLoS Comput Biol 7:e1002195.
- Mistry J et al. (2021). Pfam: The protein families database in 2021. Nucleic Acids Res 49:D412–D419.
- Johnson LS et al. (2010). Hidden Markov model speed heuristic and iterative HMM search procedure. BMC Bioinformatics 11:431.
- Paysan-Lafosse T et al. (2023). InterPro in 2022. Nucleic Acids Res 51:D419–D427.

# HMMER in Practice: hmmscan, phmmer, jackhmmer, and BLAST Comparison

In the previous two notebooks we looked at (1) the **theory** of profile HMMs and (2) how to **build** a profile HMM from a multiple sequence alignment with `hmmbuild`. This notebook focuses on the most common *practical* use cases of HMMER:

1. **`hmmscan`** — annotate a protein with the Pfam database of ~20,000 domain HMMs.
2. **`phmmer`** — a BLAST-like single-query search that builds a profile on-the-fly.
3. **`jackhmmer`** — iterative profile search, analogous to PSI-BLAST.
4. **BLAST vs. HMMER** — the classic "remote homology" showdown on the plant leghemoglobin vs. human hemoglobin alpha case.

### What you need before running this notebook

- HMMER installed on your `$PATH` (`hmmscan`, `phmmer`, `jackhmmer`, `hmmpress`). On Ubuntu/WSL: `sudo apt install hmmer`.
- The Pfam-A profile HMM database (`Pfam-A.hmm`) downloaded from the EBI. See the config cell below for the download URL.
- Internet access for UniProt FASTA fetches and online BLAST (`NCBIWWW.qblast`).
- Python packages: `biopython`, `numpy`, `pandas`, `matplotlib`.

In [ ]:
import os

# ─── Adjustable paths — edit for your environment ────────────────────────────
PFAM_HMM_PATH = "/mnt/c/Users/onurs/Downloads/Pfam-A.hmm"
WORK_DIR = "hmmer_search_work"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)

# Check Pfam availability and print a clear message
if os.path.exists(PFAM_HMM_PATH):
    size_mb = os.path.getsize(PFAM_HMM_PATH) / 1e6
    print(f"Pfam-A.hmm found at {PFAM_HMM_PATH} ({size_mb:.1f} MB)")
    # Check if pressed
    if os.path.exists(PFAM_HMM_PATH + ".h3i"):
        print("Pfam is already hmmpress'd and ready for hmmscan.")
    else:
        print("Pfam found but not hmmpress'd. Will run hmmpress in Section 2.")
else:
    print(f"WARNING: Pfam-A.hmm NOT found at {PFAM_HMM_PATH}")
    print("Download from: https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.hmm.gz")
    print("After downloading, gunzip and update PFAM_HMM_PATH above.")

In [ ]:
import subprocess
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.Blast import NCBIWWW, NCBIXML

print("Imports complete.")

---

## Section 1: Fetch Query Sequences from UniProt

We will use five protein sequences chosen to illustrate different HMMER use cases:

| UniProt ID | Name | Length | Why chosen |
|------------|------|-------:|-----------|
| **P04637** | Human p53 (tumor suppressor) | 393 aa | Multi-domain protein with a long disordered region — great for `hmmscan` domain annotation |
| **P02945** | Bacteriorhodopsin (*Halobacterium salinarum*) | 262 aa | Classical 7-TM membrane protein; distinct Pfam family |
| **P01308** | Human insulin (preproinsulin) | 110 aa | Very short, one Pfam family — simple hmmscan demo |
| **P69905** | Human hemoglobin alpha subunit | 142 aa | Globin fold — query for phmmer |
| **P02240** | Lupin leghemoglobin (*Lupinus luteus*) | 153 aa | Plant globin sharing only ~15% identity with human Hb — the classic **remote homology** test case for BLAST vs. HMMER |

We fetch FASTA records directly from the UniProt REST API and cache them in the working directory.

In [ ]:
def fetch_uniprot_fasta(accession, work_dir):
    """Download a single UniProt FASTA record. Cached: skipped if already on disk."""
    url = f"https://www.uniprot.org/uniprot/{accession}.fasta"
    out_path = os.path.join(work_dir, f"{accession}.fa")
    if not os.path.exists(out_path):
        print(f"Fetching {accession} from UniProt ...")
        urllib.request.urlretrieve(url, out_path)
    else:
        print(f"Using cached {out_path}")
    return out_path


uniprot_ids = {
    'P04637': 'p53 (human tumor suppressor)',
    'P02945': 'Bacteriorhodopsin (archaeal 7TM)',
    'P01308': 'Insulin (human)',
    'P69905': 'Hemoglobin alpha (human)',
    'P02240': 'Leghemoglobin (Lupinus luteus)',
}

fasta_paths = {}
for uid, description in uniprot_ids.items():
    fasta_paths[uid] = fetch_uniprot_fasta(uid, WORK_DIR)

print()
print("All five FASTA files present in", WORK_DIR)

In [ ]:
# Load each FASTA with Biopython and print ID, description, sequence length
header = f"{'Accession':<10}  {'Length':>6}  {'Description':<45}  First 30 residues"
print(header)
print("-" * 110)
for uid, description in uniprot_ids.items():
    record = SeqIO.read(fasta_paths[uid], "fasta")
    print(f"{uid:<10}  {len(record.seq):>6}  {description:<45}  {str(record.seq)[:30]}...")


In [ ]:
# Build a combined multi-FASTA containing all five sequences.
# We'll use this as the "reference database" for phmmer and jackhmmer later.

combined_fasta = os.path.join(WORK_DIR, "all_queries.fa")

with open(combined_fasta, "w") as out:
    for uid in uniprot_ids:
        with open(fasta_paths[uid]) as f:
            out.write(f.read())
            # Ensure a newline between records
            if not out.tell() or True:
                pass

# Verify
records = list(SeqIO.parse(combined_fasta, "fasta"))
print(f"Combined multi-FASTA: {combined_fasta}")
print(f"Contains {len(records)} records:")
for r in records:
    print(f"  {r.id:<30}  length={len(r.seq)}")

---

## Section 2: hmmpress Pfam (if needed)

`hmmscan` requires the HMM database to be **pre-compressed into binary index files** (`.h3i`, `.h3m`, `.h3f`, `.h3p`). This is done once per database with `hmmpress`. For Pfam-A.hmm (around 1.5 GB of HMMs), `hmmpress` typically takes **1–3 minutes** and produces several GB of index files.

If your `Pfam-A.hmm` already has the `.h3i` companion file next to it, this step is a no-op.

In [ ]:
# Check which Pfam index files are already present
needed_extensions = ['.h3i', '.h3m', '.h3f', '.h3p']
missing = [ext for ext in needed_extensions
           if not os.path.exists(PFAM_HMM_PATH + ext)]

if not os.path.exists(PFAM_HMM_PATH):
    print("SKIPPING Section 2: Pfam-A.hmm not available.")
    print("Update PFAM_HMM_PATH in the config cell above and re-run.")
elif len(missing) == 0:
    print("All four hmmpress binary files are present. Skipping hmmpress.")
    for ext in needed_extensions:
        fsize_mb = os.path.getsize(PFAM_HMM_PATH + ext) / 1e6
        print(f"  {PFAM_HMM_PATH + ext:<70} {fsize_mb:>8.1f} MB")
else:
    print(f"Missing hmmpress index files: {missing}")
    print("Running hmmpress (this may take 1-3 minutes) ...")
    result = subprocess.run(
        ['hmmpress', PFAM_HMM_PATH],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("ERROR: hmmpress failed.")
        print(result.stderr)
    else:
        print("hmmpress completed successfully.")
        print(result.stdout)

---

## Section 3: hmmscan — Domain Annotation of p53

`hmmscan` takes **one query sequence** and searches it against a **database of HMMs** (here, all of Pfam). Output tells us *"which known domains does this protein contain, and where?"*.

This is the most common practical HMMER use case: given a new protein sequence (from a genome assembly, a crystallography project, etc.), quickly annotate all its known domains.

For human p53 we expect three well-known Pfam domains:
- **P53_TAD** — the N-terminal transactivation domain (flexible)
- **P53** — the central DNA-binding domain (the structured core)
- **P53_tetramer** — the oligomerisation/tetramerisation domain near the C-terminus

There are large regions of p53 that are **intrinsically disordered** and are therefore *not* covered by any Pfam domain.

In [ ]:
# Run hmmscan on p53 against Pfam-A.hmm
# --domtblout produces a per-domain tab-delimited hits table
# --cpu sets the number of threads (4 is a good default)

p53_domtbl = os.path.join(WORK_DIR, "p53_domains.tbl")

print("Running hmmscan on p53 against Pfam-A.hmm (may take ~30-60 seconds) ...")
result = subprocess.run(
    ['hmmscan', '--domtblout', p53_domtbl, '--cpu', '4',
     PFAM_HMM_PATH, fasta_paths['P04637']],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    # Show the beginning of the human-readable stdout report
    print(result.stdout[:3000])
    print("...")
    print(f"\nFull per-domain table written to: {p53_domtbl}")

In [ ]:
def parse_domtblout(path):
    """Parse an hmmscan --domtblout file into a pandas DataFrame.

    The domtblout format has 22 fixed whitespace-delimited fields; the 23rd
    field onwards is the free-form description (may contain spaces).
    We use split(maxsplit=22) to preserve the description intact.
    """
    rows = []
    with open(path) as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            fields = line.split(maxsplit=22)
            rows.append({
                'target_name':  fields[0],
                'target_acc':   fields[1],
                'query_name':   fields[3],
                'i_evalue':     float(fields[12]),
                'score':        float(fields[13]),
                'hmm_from':     int(fields[15]),
                'hmm_to':       int(fields[16]),
                'ali_from':     int(fields[17]),
                'ali_to':       int(fields[18]),
                'env_from':     int(fields[19]),
                'env_to':       int(fields[20]),
                'description':  fields[22] if len(fields) > 22 else '',
            })
    return pd.DataFrame(rows)


p53_hits = parse_domtblout(p53_domtbl).sort_values('i_evalue').reset_index(drop=True)

print(f"Total p53 Pfam hits reported: {len(p53_hits)}")
print()
print("Top hits (sorted by independent E-value):")
display_cols = ['target_name', 'target_acc', 'ali_from', 'ali_to',
                'i_evalue', 'score']
print(p53_hits[display_cols].to_string(index=False))

In [ ]:
# Plot p53 domain architecture
# - Baseline bar from 0 to 393 in light gray (total p53 length)
# - Significant hits (i_evalue < 0.01) drawn as colored rectangles from ali_from to ali_to
# - Label each with target_name

p53_length = 393
significant = p53_hits[p53_hits['i_evalue'] < 0.01].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 3.5))

# Backbone bar
ax.barh(y=1, width=p53_length, left=0, height=0.15,
        color='#d0d0d0', edgecolor='#999999', label='p53 backbone')

# Choose a color palette
palette = ['#2a7f8a', '#e07070', '#6a8f3a', '#c06ea8', '#d4a017', '#4a6fa5']

for i, row in significant.iterrows():
    color = palette[i % len(palette)]
    width = row['ali_to'] - row['ali_from'] + 1
    ax.barh(y=1, width=width, left=row['ali_from'], height=0.5,
            color=color, edgecolor='black', linewidth=0.6)
    # Label
    ax.text(
        x=(row['ali_from'] + row['ali_to']) / 2,
        y=1,
        s=row['target_name'],
        ha='center', va='center', fontsize=9, fontweight='bold', color='white'
    )
    # Coords annotation under the bar
    ax.text(
        x=(row['ali_from'] + row['ali_to']) / 2,
        y=0.55,
        s=f"{row['ali_from']}-{row['ali_to']}",
        ha='center', va='top', fontsize=8, color='black'
    )

ax.set_xlim(-5, p53_length + 5)
ax.set_ylim(0.3, 1.5)
ax.set_xlabel("Residue position", fontsize=11)
ax.set_yticks([])
ax.set_title("Domain architecture of human p53 (P04637, 393 aa)", fontsize=12)

# Ticks every 50 residues
ax.set_xticks(range(0, p53_length + 1, 50))

plt.tight_layout()
plt.show()

print()
print("Regions of p53 NOT covered by any significant Pfam hit:")
covered = np.zeros(p53_length + 1, dtype=bool)
for _, row in significant.iterrows():
    covered[row['ali_from']:row['ali_to'] + 1] = True
# Report runs of uncovered residues
i = 1
while i <= p53_length:
    if not covered[i]:
        start = i
        while i <= p53_length and not covered[i]:
            i += 1
        end = i - 1
        print(f"  residues {start:>3d}-{end:<3d}  (length {end - start + 1})")
    else:
        i += 1

**Question:** What are the regions of p53 *NOT* covered by any Pfam domain? What are those regions likely to be structurally?

*Hint:* Large uncovered regions in well-studied proteins often correspond to **intrinsically disordered regions (IDRs)** — flexible linkers without a fixed three-dimensional structure. For p53, the N-terminal transactivation portion and the long C-terminal regulatory tail are classic IDRs. Pfam domains are built from structurally stable regions, so disordered regions are deliberately excluded from family models.

---

## Section 4: phmmer — Quick Sequence vs. Sequence Database Search

`phmmer` is HMMER's **most BLAST-like tool**. Given a single query sequence and a target sequence database, `phmmer` internally:

1. builds a profile HMM from the query sequence (using position-independent background priors),
2. searches that profile against the target database,
3. reports the hits with E-values.

Unlike BLAST, `phmmer` uses the full HMM forward/Viterbi algorithm under the hood — so it is already a small step toward HMM sensitivity even though we start from a single sequence.

We will search **human hemoglobin alpha (P69905)** against our 5-sequence reference database (`all_queries.fa`).

In [ ]:
# Run phmmer: query = P69905 (Hb alpha), database = all_queries.fa
# --tblout  = per-sequence hit table (one line per full-sequence hit)
# --cpu 4   = use 4 threads

hba_phmmer_tbl = os.path.join(WORK_DIR, "hba_phmmer.tbl")

print("Running phmmer: P69905 (Hb alpha) vs. all_queries.fa ...")
result = subprocess.run(
    ['phmmer', '--tblout', hba_phmmer_tbl, '--cpu', '4',
     fasta_paths['P69905'], combined_fasta],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    # Print the top of the human-readable report
    print(result.stdout[:2500])
    print(f"\n(full tblout at {hba_phmmer_tbl})")

In [ ]:
def parse_tblout(path):
    """Parse an hmmsearch/phmmer --tblout file into a pandas DataFrame."""
    rows = []
    with open(path) as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            fields = line.split()
            rows.append({
                'target':              fields[0],
                'target_accession':    fields[1],
                'query':               fields[2],
                'query_accession':     fields[3],
                'full_seq_evalue':     float(fields[4]),
                'full_seq_score':      float(fields[5]),
                'full_seq_bias':       float(fields[6]),
                'best_domain_evalue':  float(fields[7]),
                'best_domain_score':   float(fields[8]),
            })
    return pd.DataFrame(rows)


hba_phmmer_hits = parse_tblout(hba_phmmer_tbl).sort_values('full_seq_evalue').reset_index(drop=True)

print("phmmer hits of P69905 (Hb alpha) vs. all_queries.fa:")
print(hba_phmmer_hits[['target', 'full_seq_evalue', 'full_seq_score',
                       'best_domain_evalue', 'best_domain_score']].to_string(index=False))

**Question:** Which sequences does `phmmer` report as significant hits? Which are expected vs. surprising?

*Hint:* The query matches itself with an essentially zero E-value (trivial). Any **other** globin sequence (myoglobin-like, or the plant leghemoglobin P02240) should also be detected with at least a moderately small E-value if they are close enough. Unrelated sequences (p53, insulin, bacteriorhodopsin) should NOT appear in the hits table — or should appear only with very poor E-values that would be filtered out by the default inclusion threshold (E=0.01).

---

## Section 5: BLAST vs. HMMER — The Remote Homology Advantage

A core claim of HMMER (and of profile methods in general) is that they detect **remote homologs** better than pairwise tools like BLAST. This is because a profile captures *which positions are conserved* and *which tolerate substitutions*, so it can recognise distant relatives that share the key positions even when overall sequence identity drops below 20%.

### Test case: lupin leghemoglobin vs. human hemoglobin alpha

- **P02240** (lupin leghemoglobin) shares roughly **15% sequence identity** with human hemoglobin alpha (P69905).
- They share the same **globin fold** and the same **heme-binding function**.
- At ~15% identity, BLAST is notoriously unreliable — the alignments are often not statistically significant, even though the two proteins are genuinely homologous.

We will now:
1. Run `BLASTP` **online** against SwissProt using lupin leghemoglobin as the query.
2. Run `phmmer` with the same query against our 5-sequence database.
3. Ask: does each method find human hemoglobin alpha?

In [ ]:
# Run online BLAST (if not already cached).
# Online BLAST takes 30-60 seconds per call, so we cache the XML response.

legh_record = SeqIO.read(fasta_paths['P02240'], 'fasta')
print(f"Query: {legh_record.id}, length {len(legh_record.seq)}")

blast_path = os.path.join(WORK_DIR, "legh_blast.xml")

if os.path.exists(blast_path):
    print(f"Using cached BLAST XML at {blast_path}")
else:
    print("Running online BLAST against SwissProt (this may take ~30-60 seconds) ...")
    result_handle = NCBIWWW.qblast(
        'blastp', 'swissprot', legh_record.seq,
        hitlist_size=20, expect=10.0
    )
    blast_xml = result_handle.read()
    result_handle.close()
    with open(blast_path, 'w') as f:
        f.write(blast_xml)
    print(f"BLAST XML saved to {blast_path}")

# Parse the XML
with open(blast_path) as f:
    blast_record = NCBIXML.read(f)

blast_rows = []
for alignment in blast_record.alignments[:20]:
    best_hsp = min(alignment.hsps, key=lambda h: h.expect)
    blast_rows.append({
        'title':        alignment.title[:70],
        'length':       alignment.length,
        'evalue':       best_hsp.expect,
        'score':        best_hsp.score,
        'identity_pct': 100 * best_hsp.identities / best_hsp.align_length,
    })

blast_df = pd.DataFrame(blast_rows)
print("\nBLAST top 20 hits:")
print(blast_df.to_string(index=False))

In [ ]:
# Now run phmmer with leghemoglobin against all_queries.fa
legh_phmmer_tbl = os.path.join(WORK_DIR, "legh_phmmer.tbl")

print("Running phmmer: P02240 (leghemoglobin) vs. all_queries.fa ...")
result = subprocess.run(
    ['phmmer', '--tblout', legh_phmmer_tbl, '--cpu', '4',
     fasta_paths['P02240'], combined_fasta],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("ERROR:", result.stderr)

legh_phmmer_hits = parse_tblout(legh_phmmer_tbl).sort_values('full_seq_evalue').reset_index(drop=True)

print()
print("phmmer hits of P02240 (leghemoglobin) vs. all_queries.fa:")
print(legh_phmmer_hits[['target', 'full_seq_evalue', 'full_seq_score',
                        'best_domain_evalue', 'best_domain_score']].to_string(index=False))

### Why does BLAST struggle vs. HMMER on remote homologs?

BLAST uses a **single substitution matrix** (BLOSUM62 by default) applied uniformly to every position. It essentially asks: *"given standard amino acid substitution frequencies, how likely is this alignment by chance?"*

HMMER uses a **profile** (even a profile built on-the-fly from one sequence, as in `phmmer`) where **position-specific transition probabilities** mean insertions and deletions at structurally conserved regions are penalised more heavily than at variable loops. More importantly, `phmmer` applies the HMMER3 forward algorithm and its improved E-value statistics, which yield higher sensitivity and better-calibrated significance at low identity than BLAST's Karlin-Altschul statistics in the low-identity regime.

The net effect: at identities in the 15–25% range — the "twilight zone" of homology detection — HMMER routinely finds hits that BLAST misses entirely.

In [ ]:
# Comparison plot: BLAST E-values vs. phmmer E-values for top hits
# Highlight P69905 (human Hb alpha) in both if present

fig, (ax_blast, ax_phmmer) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: BLAST E-values (top 10) ──
blast_top = blast_df.head(10).copy()
blast_top['neg_log_e'] = -np.log10(blast_top['evalue'].replace(0, 1e-300))

# Color: highlight any row whose title mentions HBA or hemoglobin alpha
def is_human_hba_title(title):
    t = title.upper()
    return ('HBA_HUMAN' in t) or ('P69905' in t)

blast_colors = ['#e07070' if is_human_hba_title(t) else '#2a7f8a'
                for t in blast_top['title']]

ax_blast.barh(range(len(blast_top)), blast_top['neg_log_e'],
              color=blast_colors, edgecolor='black', linewidth=0.4)
ax_blast.set_yticks(range(len(blast_top)))
ax_blast.set_yticklabels([t[:45] for t in blast_top['title']], fontsize=8)
ax_blast.invert_yaxis()
ax_blast.set_xlabel("-log10(E-value)", fontsize=11)
ax_blast.set_title("BLAST: top 10 hits\n(red = human Hb alpha)", fontsize=11)
ax_blast.grid(axis='x', alpha=0.3)

# ── Right: phmmer E-values ──
phmmer_top = legh_phmmer_hits.copy()
phmmer_top['neg_log_e'] = -np.log10(phmmer_top['full_seq_evalue'].replace(0, 1e-300))

phmmer_colors = ['#e07070' if 'P69905' in t else '#2a7f8a'
                 for t in phmmer_top['target']]

ax_phmmer.barh(range(len(phmmer_top)), phmmer_top['neg_log_e'],
               color=phmmer_colors, edgecolor='black', linewidth=0.4)
ax_phmmer.set_yticks(range(len(phmmer_top)))
ax_phmmer.set_yticklabels(phmmer_top['target'], fontsize=9)
ax_phmmer.invert_yaxis()
ax_phmmer.set_xlabel("-log10(E-value)", fontsize=11)
ax_phmmer.set_title("phmmer: all hits\n(red = human Hb alpha P69905)", fontsize=11)
ax_phmmer.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Textual verdict
print()
blast_has_hba = any(is_human_hba_title(t) for t in blast_df['title'])
phmmer_has_hba = any('P69905' in t for t in legh_phmmer_hits['target'])

print(f"BLAST found human Hb alpha (HBA_HUMAN / P69905): {blast_has_hba}")
print(f"phmmer found human Hb alpha (P69905):            {phmmer_has_hba}")

**Question:** Does BLAST find human hemoglobin alpha when searching with lupin leghemoglobin? Does phmmer? Why might they differ at low sequence identity?

*Hint:* BLAST relies on finding ungapped seed words (typically length 3 for protein) that score above a threshold under BLOSUM62. At ~15% identity, there are very few such seeds, so the search often fails to extend into a significant HSP. HMMER's forward-algorithm statistics accumulate signal from *all* paths through the profile simultaneously, so many weak matches at conserved positions can together produce a statistically significant score even when no single local region is BLAST-like.

---

## Section 6: jackhmmer — Iterative Search

`jackhmmer` is HMMER's iterative profile search, conceptually similar to **PSI-BLAST**:

1. **Iteration 1**: start with a single sequence query, build a profile from it, search the database.
2. **Iteration 2**: take all significant hits from iteration 1, build a multiple alignment, re-estimate the profile, search again.
3. **Iteration 3, 4, …**: repeat until the set of significant hits **converges** (no new sequences are added).

Each iteration *expands* the profile with what it has found, so the model becomes more sensitive to distant relatives at each step. The trade-off: if a false positive sneaks in during early iterations, the profile drifts and subsequent iterations can amplify the error (*profile drift*).

We will run `jackhmmer` for 3 iterations with lupin leghemoglobin as the seed, searching our 5-sequence database.

In [ ]:
# Run jackhmmer with up to 3 iterations
jack_out = os.path.join(WORK_DIR, "legh_jackhmmer.out")
jack_tbl = os.path.join(WORK_DIR, "legh_jackhmmer.tbl")

print("Running jackhmmer (N=3 iterations): P02240 vs. all_queries.fa ...")
result = subprocess.run(
    ['jackhmmer', '-N', '3', '--tblout', jack_tbl,
     fasta_paths['P02240'], combined_fasta],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("ERROR:", result.stderr)

# Save the full stdout text for reference
with open(jack_out, 'w') as f:
    f.write(result.stdout)

print(f"jackhmmer stdout saved to {jack_out} ({len(result.stdout)} chars)")
print(f"Final tblout saved to {jack_tbl}")

In [ ]:
# Parse jackhmmer stdout to count hits per iteration.
# jackhmmer prints '@@' marker blocks at the start of each iteration, with lines like:
#   @@ Round:                  1
#   @@ Included in MSA:        N subsequences (query + N-1 subseqs from N-1 targets)
#
# We extract 'Round' numbers and the 'Included in MSA' subseq counts as a simple
# per-iteration summary.

import re

round_pattern   = re.compile(r'@@ Round:\s+(\d+)')
include_pattern = re.compile(r'@@ Included in MSA:\s+(\d+)\s+subsequences')

with open(jack_out) as f:
    text = f.read()

rounds = round_pattern.findall(text)
included = include_pattern.findall(text)

print(f"Rounds found in jackhmmer output: {rounds}")
print(f"'Included in MSA' counts per round: {included}")
print()

if included and len(included) >= 2:
    if included[-1] == included[-2]:
        print(f"CONVERGED: the last two iterations had the same hit set ({included[-1]} subsequences).")
    else:
        print("Did NOT converge: the hit set changed between the last two iterations.")
else:
    print("Not enough iterations completed to judge convergence.")

# Also show the final tblout contents
print()
print("Final-iteration per-sequence hits (from --tblout):")
final_hits = parse_tblout(jack_tbl).sort_values('full_seq_evalue')
print(final_hits[['target', 'full_seq_evalue', 'full_seq_score']].to_string(index=False))

**Question:** Did `jackhmmer` converge? What does convergence mean in the context of iterative profile search?

*Hint:* **Convergence** means two consecutive iterations return the **same set of significant hits** — adding more iterations cannot bring in new sequences, so the profile has stabilised. In practice, convergence in a small database is reached very quickly (often at iteration 2 or 3). In large databases like UniRef, jackhmmer may run for 5 or more iterations. The danger with running too many iterations is **profile drift**: a false positive included in an early iteration drags the profile toward that sequence's features, enabling even more false positives in later rounds.

---

## Section 7: hmmscan on Insulin and Bacteriorhodopsin

Now we repeat the `hmmscan` workflow from Section 3 on two very different proteins:

- **P01308** (insulin) — very short, one main Pfam family (`Insulin`).
- **P02945** (bacteriorhodopsin) — a 7-transmembrane protein with the `Bac_rhodopsin` family.

This shows how the same single command annotates proteins across wildly different biological functions.

In [ ]:
# Reusable domain-architecture plot function that mirrors the p53 plot

def plot_domain_architecture(hits_df, total_length, protein_name, protein_accession):
    """Draw horizontal domain architecture from a parsed domtblout DataFrame."""
    significant = hits_df[hits_df['i_evalue'] < 0.01].reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(14, 3.0))
    ax.barh(y=1, width=total_length, left=0, height=0.15,
            color='#d0d0d0', edgecolor='#999999')

    palette = ['#2a7f8a', '#e07070', '#6a8f3a', '#c06ea8', '#d4a017', '#4a6fa5']

    for i, row in significant.iterrows():
        color = palette[i % len(palette)]
        width = row['ali_to'] - row['ali_from'] + 1
        ax.barh(y=1, width=width, left=row['ali_from'], height=0.5,
                color=color, edgecolor='black', linewidth=0.6)
        ax.text(
            x=(row['ali_from'] + row['ali_to']) / 2, y=1,
            s=row['target_name'],
            ha='center', va='center', fontsize=9, fontweight='bold', color='white'
        )
        ax.text(
            x=(row['ali_from'] + row['ali_to']) / 2, y=0.55,
            s=f"{row['ali_from']}-{row['ali_to']}",
            ha='center', va='top', fontsize=8
        )

    ax.set_xlim(-5, total_length + 5)
    ax.set_ylim(0.3, 1.5)
    ax.set_xlabel("Residue position", fontsize=11)
    ax.set_yticks([])
    ax.set_title(f"Domain architecture of {protein_name} ({protein_accession}, {total_length} aa)",
                 fontsize=12)

    # Sensible tick spacing
    step = max(10, total_length // 10)
    ax.set_xticks(range(0, total_length + 1, step))

    plt.tight_layout()
    plt.show()


# ── Insulin ──
ins_record = SeqIO.read(fasta_paths['P01308'], 'fasta')
insulin_length = len(ins_record.seq)
ins_domtbl = os.path.join(WORK_DIR, "insulin_domains.tbl")

print("Running hmmscan on insulin (P01308) ...")
result = subprocess.run(
    ['hmmscan', '--domtblout', ins_domtbl, '--cpu', '4',
     PFAM_HMM_PATH, fasta_paths['P01308']],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    ins_hits = parse_domtblout(ins_domtbl).sort_values('i_evalue')
    print("Insulin Pfam hits:")
    print(ins_hits[['target_name', 'target_acc', 'ali_from', 'ali_to',
                    'i_evalue', 'score']].to_string(index=False))
    plot_domain_architecture(ins_hits, insulin_length,
                             "Insulin", "P01308")

In [ ]:
# ── Bacteriorhodopsin ──
bac_record = SeqIO.read(fasta_paths['P02945'], 'fasta')
bac_length = len(bac_record.seq)
bac_domtbl = os.path.join(WORK_DIR, "bacrhod_domains.tbl")

print("Running hmmscan on bacteriorhodopsin (P02945) ...")
result = subprocess.run(
    ['hmmscan', '--domtblout', bac_domtbl, '--cpu', '4',
     PFAM_HMM_PATH, fasta_paths['P02945']],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    bac_hits = parse_domtblout(bac_domtbl).sort_values('i_evalue')
    print("Bacteriorhodopsin Pfam hits:")
    print(bac_hits[['target_name', 'target_acc', 'ali_from', 'ali_to',
                    'i_evalue', 'score']].to_string(index=False))
    plot_domain_architecture(bac_hits, bac_length,
                             "Bacteriorhodopsin", "P02945")

**Stretch exercise:** Fetch a protein sequence of your choice from UniProt (edit the UniProt accession below) and run `hmmscan` on it. What domains does it contain? How many disordered regions (not covered by any Pfam domain) are there?

Useful starting points:
- **P53_HUMAN** (P04637) — already done
- **BRCA1_HUMAN** (P38398) — large multi-domain protein with RING, BRCT domains
- **TITIN_HUMAN** (Q8WZ42) — very long protein with many repeated Ig-like domains
- **CFTR_HUMAN** (P13569) — ABC transporter, multiple domains

---

## Section 8: E-value Interpretation

HMMER uses **two E-value thresholds** in its search pipeline:

| Threshold | Flag | Default | Purpose |
|-----------|------|---------|---------|
| Reporting | `-E` | 10.0 | Hits shown in the output report (lenient — for exploration) |
| Inclusion | `--incE` | 0.01 | Hits trusted enough to include in an MSA for iterative search (strict — for jackhmmer) |

**Why two thresholds?**
- The *reporting* threshold is deliberately permissive because a user browsing results may want to see borderline matches and judge them manually.
- The *inclusion* threshold is strict because in `jackhmmer`, any sequence included in the next iteration's MSA directly shapes the profile. Letting through a false positive at inclusion time causes **profile drift** — the profile starts looking like the spurious hit, and subsequent iterations amplify the error.

We now combine hits from p53, insulin, and bacteriorhodopsin and plot the **bit score distribution** with the two thresholds marked.

In [ ]:
# Combine all hmmscan hits from the three proteins
all_hmmscan_hits = pd.concat([p53_hits, ins_hits, bac_hits], ignore_index=True)

print(f"Combined hits table: {len(all_hmmscan_hits)} rows from p53 + insulin + bacteriorhodopsin")
print()
print("Summary of bit scores and E-values:")
print(all_hmmscan_hits[['score', 'i_evalue']].describe().to_string())

In [ ]:
# Plot histogram of bit scores, with approximate threshold lines.
# HMMER does not publish a fixed "score at E=X" because it depends on model length,
# database size, and other details. But we can use the observed data: which score value
# separates the hits at E=0.01 and E=1e-5 in this combined pool?

# Find the score at E=0.01 boundary: maximum score among hits with i_evalue >= 0.01
below_001 = all_hmmscan_hits[all_hmmscan_hits['i_evalue'] < 0.01]['score']
above_001 = all_hmmscan_hits[all_hmmscan_hits['i_evalue'] >= 0.01]['score']

if len(below_001) > 0 and len(above_001) > 0:
    score_at_001 = (below_001.min() + above_001.max()) / 2
else:
    score_at_001 = 20  # fallback

below_1e5 = all_hmmscan_hits[all_hmmscan_hits['i_evalue'] < 1e-5]['score']
above_1e5 = all_hmmscan_hits[all_hmmscan_hits['i_evalue'] >= 1e-5]['score']
if len(below_1e5) > 0 and len(above_1e5) > 0:
    score_at_1e5 = (below_1e5.min() + above_1e5.max()) / 2
else:
    score_at_1e5 = 40  # fallback

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(all_hmmscan_hits['score'], bins=30,
        color='#2a7f8a', edgecolor='black', linewidth=0.5, alpha=0.8)

ax.axvline(score_at_001, color='red', linestyle='--', linewidth=2,
           label=f'~E=0.01 threshold (score ≈ {score_at_001:.1f} bits)')
ax.axvline(score_at_1e5, color='purple', linestyle=':', linewidth=2,
           label=f'~E=1e-5 threshold (score ≈ {score_at_1e5:.1f} bits)')

ax.set_xlabel("Bit score", fontsize=11)
ax.set_ylabel("Number of hits", fontsize=11)
ax.set_title("Bit score distribution across all hmmscan hits (p53 + insulin + bacteriorhodopsin)\n"
             "with approximate E-value thresholds",
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Reporting vs. inclusion: a rule of thumb**

- **Annotating a new protein with `hmmscan`?** Look at hits with E-value < 0.01 (or equivalently, bit scores above the ~E=0.01 line in the plot). These are the domains you can confidently annotate.
- **Running `jackhmmer` to find distant homologs?** Trust *only* hits below the inclusion threshold (0.01 by default). Tighter thresholds (`--incE 1e-5`) dramatically reduce the chance of profile drift at the cost of sensitivity.
- **Fishing for remote homology manually?** Use the reporting threshold (E=10) to see everything, then inspect alignments — a hit with E=0.5 that covers a known conserved motif may still be biologically meaningful.

---

## Section 9: Resources

| Resource | URL / Reference |
|----------|-----------------|
| HMMER User Guide | http://hmmer.org/documentation.html |
| EBI HMMER web server | https://www.ebi.ac.uk/Tools/hmmer/ |
| Pfam (via InterPro) | https://www.ebi.ac.uk/interpro/entry/pfam/ |
| InterPro / InterProScan | https://www.ebi.ac.uk/interpro/ |
| Eddy SR (2011) HMMER3 | PLoS Comput Biol 7:e1002195 |